## ⚠️ Important: Data Privacy and Anonymization

Before using this tutorial with your own data, you should ensure that any sensitive or personally identifiable information (PII) has been removed or anonymized.

Because this package sends text to an LLM API for analysis, **any raw text you provide may leave your local environment** depending on your configuration. For this reason, you should **not upload identifiable or sensitive data unless it has been properly anonymized first**.

### A note on API data retention

Most companies will not use data you submit via an API (like in this tutorial) for training, and will delete the data after processing it within 30 days (e.g., OpenAI's policy: https://developers.openai.com/api/docs/guides/your-data). Policies vary by provider — check the terms of the model you select on OpenRouter — but in general, API submissions are treated more privately than chat-product submissions. This does **not** remove your responsibility to anonymize sensitive data first; it just means that, when working with properly anonymized text, API analysis is generally safe.

### Recommended approaches

If you are working with real data, we strongly recommend anonymizing it locally before running the analysis. We recommend:

- https://github.com/microsoft/presidio
- https://github.com/childmindresearch/anonymize-pii

These tools can help remove names, locations, emails, and other identifying information before any LLM processing occurs.

### If you prefer not to use real data

If you are not comfortable anonymizing your own dataset, you can use the **publicly available Reddit data provided in this repository**, which is safe to use for demonstration purposes.

---

**In short:**
✔ Use anonymized data whenever possible
✔ Be aware of your provider's API data retention policy
✔ When in doubt, use the provided sample dataset

# Open Notebook in Colab

⚠️ Reminder: Colab runs in Google's cloud, so any data you upload leaves your local machine. Make sure your data is anonymized first (see the privacy notes above).

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/tutorial_v3.ipynb)

In [ ]:
!pip install "git+https://github.com/childmindresearch/llm_tracker.git#egg=llm-tracker[tutorials]"

# LLM Tracker Tutorial — V3

**LLM Tracker** is a Python package for identifying instances of psychological constructs in qualitative text data — such as clinical documents, therapy transcripts, or open-ended survey responses — using large language models (LLMs).

The core workflow has four steps:

1. **LLM Coding** — run a language model over a set of documents to identify and extract quotes that are instances of psychological constructs defined in a codebook.
2. **Human Coding** — produce a second set of codings to compare against. In production this is real human-coded data; for testing purposes a second LLM run can serve as a stand-in.
3. **Comparison** — align the two sets of codings instance-by-instance using an LLM matcher, classifying each as a true positive, false positive, or false negative.
4. **Summary Tables** — compute pooled, per-document, and weighted performance metrics across constructs.

This V3 tutorial is split into two parts:

- **Part 1 — Dummy Data Workflow:** runs the LLM twice over the bundled sample CSV (`reddit_autism_anxiety_depression.csv`), using the second run as a stand-in for human coding. Useful for end-to-end pipeline testing without needing real human data.
- **Part 2 — Real Human Data Workflow:** the typical research workflow, where the LLM is run once and compared against externally produced human-coded data loaded from a CSV/xlsx file.

**Prerequisites:**
- An [OpenRouter](https://openrouter.ai) account and API key. OpenRouter provides access to a wide range of models (GPT-4o, Gemini, Claude, Llama, etc.) through a single API. Create an account, add credits, and generate a key at https://openrouter.ai/settings/keys.
- The sample data and codebook bundled with this repo (used in Part 1).
- For Part 2 only: your own human-coded export (CSV/xlsx).

## Imports

Import the core classes and functions used throughout this tutorial:
- `LLMTrackerAnalyzer` — runs LLM coding over a directory of document files or a CSV
- `LLMTrackerComparer` — aligns and compares two sets of codings (human vs LLM)
- `format_comparison_table` — converts raw comparison output into a row-level DataFrame
- `compute_summary_tables` — computes per-document, concatenated, and weighted summary metrics
- `format_concatenated`, `format_per_interview`, `format_weighted_summary` — pretty-print the three summary tables
- `load_human_coding` — load human-coded data from a CSV/xlsx (Dedoose-compatible defaults)
- `validate_against_codebook` — sanity-check that human-coded constructs all match the codebook

In [1]:
import pandas as pd
from llm_tracker import LLMTrackerAnalyzer
from llm_tracker.comparison import (
    LLMTrackerComparer,
    compute_summary_tables,
    format_comparison_table,
    format_concatenated,
    format_per_interview,
    format_weighted_summary,
)
from llm_tracker.file_handlers import load_human_coding, validate_against_codebook
from llm_tracker.utils import format_coding_table

/Users/freymon.perez/Projects/GitHub/llm_tracker/src/llm_tracker/models.py:45: UserWarning: Field name "construct" in "ConstructInstance" shadows an attribute in parent "BaseModel"
  class ConstructInstance(BaseModel):


## Paths and Configuration

Set these four values before running the rest of the tutorial. The sample CSV and codebook are bundled with the repo.

- `api_key` — your OpenRouter API key, **or** the path to a `.env` file containing it. **Important:** OpenRouter API keys are case-sensitive — copy the key exactly as shown in your OpenRouter dashboard.
- `llm_model` — any model listed at https://openrouter.ai/models. The same model is used for both LLM coding and the comparison-matching step.
- `csv_path` — the input CSV (one document per row).
- `codebook_path` — JSON file defining the constructs the LLM should look for.

In [15]:
api_key = ""
llm_model = ""
input_data = ""
codebook_path = ""


---
# Part 0 - Text Descriptives.(Optional)

Before we begin with the pipline we'll get some general information about the text we're working with. We'll use https://github.com/HLasse/TextDescriptives/tree/main?tab=readme-ov-file to accomplish this. 

Text Descriptives will give us a number of metrics to gain an understanding of the structure of our text data. Please note that Text descriptives works on a per-document basis, and does not accept directories of documents. As such if you have several documents you want metrics on you will need to run them individually or concatenate them prior to using the package.

For the purposes of this tutorial we'll use the reddit sample data here, but feel free to change the path to your own data.

In [2]:
# Download the small English spaCy model (used by TextDescriptives for tokenization & parsing)
# If you'd like to use bigger models than en_core_web_sm (12 MB) optiosn include: 
# en_core_web_md (40 MB); en_core_web_lg (560 MB); en_core_web_trf (440 MB- Most accurate but slowest)
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 30.2 MB/s  0:00:00m0:00:010:01

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import textdescriptives 
reddit_data =  pd.read_csv("sample_data/reddit_autism_anxiety_depression.csv")

reddit_data_metrics = textdescriptives.extract_metrics(
    text = reddit_data["post"].tolist(),
    spacy_model = "en_core_web_sm",
    metrics = ["descriptive_stats",
    "readability",
    "dependency_distance",
    "pos_proportions",
    "coherence",
    "quality",
    "information_theory"]
) 

/Users/freymon.perez/Projects/GitHub/llm_tracker/.venv/lib/python3.12/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/Users/freymon.perez/Projects/GitHub/llm_tracker/.venv/lib/python3.12/site-packages/textdescriptives/components/coherence.py:44: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Span.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your 

In [3]:
reddit_data_metrics

,text,dependency_distance_mean,dependency_distance_std,prop_adjacent_dependency_relation_mean,prop_adjacent_dependency_relation_std,pos_prop_ADJ,pos_prop_ADP,pos_prop_ADV,pos_prop_AUX,pos_prop_CCONJ,...,duplicate_ngram_chr_fraction_5,duplicate_ngram_chr_fraction_6,duplicate_ngram_chr_fraction_7,duplicate_ngram_chr_fraction_8,duplicate_ngram_chr_fraction_9,duplicate_ngram_chr_fraction_10,top_ngram_chr_fraction_2,top_ngram_chr_fraction_3,top_ngram_chr_fraction_4,oov_ratio
0,How many people here work and had a hard time ...,2.632204,0.534885,0.422030,0.086482,0.059361,0.063927,0.059361,0.073059,0.068493,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.012295,0.000000,0.000000,None
1,How to deal with brothers girlfriend I'm 16 yr...,2.254639,1.007319,0.373999,0.135526,0.054945,0.093407,0.098901,0.098901,0.054945,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.029777,0.026055,0.000000,None
2,Break-in anxiety advice desperately needed So ...,2.463340,0.809090,0.405302,0.126944,0.051724,0.060345,0.060345,0.094828,0.043103,...,0.080769,0.080769,0.080769,0.000000,0.000000,0.000000,0.017308,0.000000,0.000000,None
3,I really cannot deal with living like this. I ...,2.631157,0.535368,0.455457,0.041598,0.083333,0.069444,0.104167,0.083333,0.027778,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.018838,0.000000,0.000000,None
4,Can someone track my flight? Flying DFW to MIA...,2.407669,0.603180,0.420095,0.062707,0.062500,0.052083,0.104167,0.083333,0.041667,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,None
5,"Um hey Not sure this is important, but I went ...",2.522556,0.048872,0.398496,0.030075,0.181818,0.060606,0.030303,0.030303,0.060606,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,None
6,Is this anxiety? I know this might be a dumb q...,2.738095,0.922753,0.342321,0.066935,0.049180,0.065574,0.073770,0.090164,0.040984,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.026408,0.000000,0.000000,None
7,HOT FLASHES AND ANXIETY ATTACK ON DAY OF EOC I...,1.500000,0.500000,0.557692,0.057692,0.058824,0.117647,0.000000,0.000000,0.117647,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,None
8,"Moving out and new job Hello there, this is my...",3.177868,1.002912,0.461054,0.077385,0.050209,0.083682,0.062762,0.108787,0.050209,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.009100,0.000000,0.000000,None
9,"One of my worst days I’ve had bad days, I’ve h...",1.962121,0.559889,0.564394,0.043848,0.079365,0.111111,0.047619,0.063492,0.031746,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,None


---

# Part 1 — Dummy Data Workflow

In this part we run the LLM twice over the same sample CSV and treat the second run as a stand-in for human coding. This lets you test the full pipeline end-to-end before you have real human-coded data to compare against.

Because the same model is run twice on the same documents, agreement will be high but not perfect — the LLM introduces some variability between runs, which makes this a reasonable functional test of the whole workflow.

## Step 1 (Dummy): LLM Coding

This step runs the LLM over every row of `csv_path` and extracts quotes that are instances of each construct in the codebook. Each document is processed independently, and results are saved to a timestamped output directory so runs are never overwritten.

The output directory has the following structure:
```
LLM_coding_YYYY-MM-DD_HHMMSS/
    codings/     <- one JSON per document, containing extracted construct instances
    metadata/    <- token usage, latency, and model info per document
    errors/      <- any documents that failed, with error messages
    README.md
```

Each coding JSON contains a list of instances, where each instance includes the construct name, the extracted quote, character-level indices into the source text, and a confidence score from the LLM.

In [5]:
analyzer = LLMTrackerAnalyzer(
    api_key=api_key,
    model_name=llm_model,
)

results_llm, metadata_llm, errors_llm = analyzer.analyze_csv(
    csv_path=input_data,
    text_column="post",
    subreddit_column="subreddit",
    author_column="author",
    codebook_path=codebook_path,
    output_dir="LLM_coding",  # timestamp suffix added automatically
)

Processing [1/60]: anxiety_An0nus3r123
  ✓ Found 5 construct instances
  Progress: Successful 1/60; Errors 0/60
Processing [2/60]: anxiety_Anon1234567899754
  ✓ Found 3 construct instances
  Progress: Successful 2/60; Errors 0/60
Processing [3/60]: anxiety_vwerfel
  ✓ Found 3 construct instances
  Progress: Successful 3/60; Errors 0/60
Processing [4/60]: anxiety_LostWildOne
  ✓ Found 7 construct instances
  Progress: Successful 4/60; Errors 0/60
Processing [5/60]: anxiety_frauleinfarts
  ✓ Found 2 construct instances
  Progress: Successful 5/60; Errors 0/60
Processing [6/60]: anxiety_neo-mancer
  ✓ Found 2 construct instances
  Progress: Successful 6/60; Errors 0/60
Processing [7/60]: anxiety_4ssCr4ck
  ✓ Found 7 construct instances
  Progress: Successful 7/60; Errors 0/60
Processing [8/60]: anxiety_NeptuneOW
  ✓ Found 3 construct instances
  Progress: Successful 8/60; Errors 0/60
Processing [9/60]: anxiety_zhvng666
  ✓ Found 7 construct instances
  Progress: Successful 9/60; Errors 0/

### Alternative: directory of `.txt` files

If your data is a folder of `.txt` files (one document per file) instead of a CSV, use `analyze_directory(...)` instead of `analyze_csv(...)`. Same return shape, same downstream pipeline. Each filename (without extension) becomes the `document_id`. The cell below is left commented out — uncomment and set `input_dir` if that's your input format.

In [ ]:
# Alternative: analyze a directory of .txt files instead of a CSV.

# input_dir = ""  # path to your directory of .txt files

# results_llm, metadata_llm, errors_llm = analyzer.analyze_directory(
#     input_dir=input_dir,
#     codebook_path=codebook_path,
#     output_dir="LLM_coding",  # timestamp suffix added automatically
# )

The analyzer always returns and saves three dictionaries:
- `results` — what the LLM actually coded
- `metadata` — information about how the LLM was prompted (tokens, latency, model)
- `errors` — information on any calls that failed

These are also persisted as JSON files inside the timestamped output directory.

In [6]:
format_coding_table(results_llm)

Coding Table
------------
One row per construct instance across all documents. Each row represents
a single quote identified by the LLM as an instance of a construct.

Columns:
  doc_id      : document identifier (filename or subreddit_author)
  construct   : psychological construct the instance belongs to
  quote       : exact quote extracted from the source text
  speaker_id  : speaker identifier if available in the source text
  quote_index : character-level start:end indices of the quote in the source text
  confidence  : LLM confidence score (0=not mentioned/negated, 1=indirect, 2=clear)



,doc_id,construct,quote,speaker_id,quote_index,confidence
0,anxiety_An0nus3r123,anxiety,I know my anxiety has made it hard for me to g...,None,71:138,2
1,anxiety_An0nus3r123,anxiety,Because again i know anxiety can make it hard ...,None,432:513,2
2,anxiety_An0nus3r123,shame,I'm not a interesting person,None,804:832,2
3,anxiety_An0nus3r123,shame,idk how to come off as a fun person,None,688:723,1
4,anxiety_An0nus3r123,anhedonia,i kinda just wanna do my job and leave,None,860:898,1
...,...,...,...,...,...,...
259,depression_JovialDemon01,rumination,"I just keep asking myself, what's wrong with m...",None,392:528,2
260,depression_Domihoes,depressed_mood,deep depression,None,51:66,2
261,depression_Domihoes,depressed_mood,really low,None,137:147,2
262,depression_Domihoes,depressed_mood,cry my tears dry,None,409:425,2


### Retry Failed Documents (optional)

If any documents failed — due to API errors, rate limits, or malformed responses — you can retry only the failed ones without re-processing documents that already succeeded. Uncomment the code below and update `output_dir` to point to the timestamped directory from your run above.

In [ ]:
# recovered_results, recovered_meta, remaining_errors = analyzer.retry_errors(
#     output_dir="path/to/LLM_coding_YYYY-MM-DD_HHMMSS",
#     codebook_path=codebook_path,
# )

## Step 2 (Dummy): Stand-in Human Coding via a Second LLM Run

For the comparison step we need a second set of codings. In Part 1 we generate that by running the LLM a second time over the same documents — the variability between runs gives us something non-trivial to compare against.

Variables in this section are prefixed with `dummy_human_` to make it clear this is *not* real human data.

> If you have real human codings, **skip Part 1 entirely and jump to Part 2**, which shows the typical workflow.

In [7]:
dummy_human_results, dummy_human_metadata, dummy_human_errors = analyzer.analyze_csv(
    csv_path=input_data,
    text_column="post",
    subreddit_column="subreddit",
    author_column="author",
    codebook_path=codebook_path,
    output_dir="dummy_human_coding",  # timestamp suffix added automatically
)

Processing [1/60]: anxiety_An0nus3r123
  ✓ Found 4 construct instances
  Progress: Successful 1/60; Errors 0/60
Processing [2/60]: anxiety_Anon1234567899754
  ✓ Found 2 construct instances
  Progress: Successful 2/60; Errors 0/60
Processing [3/60]: anxiety_vwerfel
  ✓ Found 3 construct instances
  Progress: Successful 3/60; Errors 0/60
Processing [4/60]: anxiety_LostWildOne
  ✓ Found 9 construct instances
  Progress: Successful 4/60; Errors 0/60
Processing [5/60]: anxiety_frauleinfarts
  ✓ Found 2 construct instances
  Progress: Successful 5/60; Errors 0/60
Processing [6/60]: anxiety_neo-mancer
  ✓ Found 2 construct instances
  Progress: Successful 6/60; Errors 0/60
Processing [7/60]: anxiety_4ssCr4ck
  ✓ Found 7 construct instances
  Progress: Successful 7/60; Errors 0/60
Processing [8/60]: anxiety_NeptuneOW
  ✓ Found 3 construct instances
  Progress: Successful 8/60; Errors 0/60
Processing [9/60]: anxiety_zhvng666
  ✓ Found 7 construct instances
  Progress: Successful 9/60; Errors 0/

In [8]:
format_coding_table(dummy_human_results)

Coding Table
------------
One row per construct instance across all documents. Each row represents
a single quote identified by the LLM as an instance of a construct.

Columns:
  doc_id      : document identifier (filename or subreddit_author)
  construct   : psychological construct the instance belongs to
  quote       : exact quote extracted from the source text
  speaker_id  : speaker identifier if available in the source text
  quote_index : character-level start:end indices of the quote in the source text
  confidence  : LLM confidence score (0=not mentioned/negated, 1=indirect, 2=clear)



,doc_id,construct,quote,speaker_id,quote_index,confidence
0,anxiety_An0nus3r123,anxiety,I know my anxiety has made it hard for me to g...,None,71:138,2
1,anxiety_An0nus3r123,anxiety,Because again i know anxiety can make it hard ...,None,432:513,2
2,anxiety_An0nus3r123,shame,"And i feel like saying watching tv, reading, f...",None,514:612,1
3,anxiety_An0nus3r123,shame,I'm not a interesting person,None,804:832,2
4,anxiety_Anon1234567899754,anxiety,I feel so stressed and uncomfortable in my own...,None,194:258,2
...,...,...,...,...,...,...
251,depression_JovialDemon01,depressed_mood,anxiety/depression medication,None,584:613,2
252,depression_Domihoes,depressed_mood,deep depression,None,51:66,2
253,depression_Domihoes,depressed_mood,lowest ive ever been in my life right now ( re...,None,93:149,2
254,depression_Domihoes,depressed_mood,cry my tears dry,None,409:425,2


## Step 3 (Dummy): Comparison

This step aligns the dummy-human and LLM codings for each document. For each construct, an LLM matcher reads all quotes from both coders and decides which dummy-human quotes and LLM quotes refer to the same passage or idea — even if the wording differs (paraphrase).

Each instance in the resulting DataFrame is classified as one of three statuses:
- **`matched`** — both coders identified this passage (True Positive, TP). Includes a `match_confidence` score from the matcher and a `span_overlap` score (Jaccard overlap of character indices).
- **`human_only`** — the (dummy) human identified this passage but the LLM missed it (False Negative, FN).
- **`llm_only`** — the LLM identified this passage but the (dummy) human missed it (False Positive, FP).

You can optionally use a different model for the matching step than for coding — for example, a more capable model for matching if accuracy is critical. Set `match_model` accordingly.

In [9]:
comparator = LLMTrackerComparer(
    api_key=api_key,
    match_model=llm_model,
)

comparison_results = comparator.compare_results(
    human_results=dummy_human_results,
    llm_results=results_llm,
    output_dir="dummy_comparison_run",  # optional; saves JSON comparison files
)

## Step 4 (Dummy): Build Summary Tables

Now that we have compared the LLM coding to the dummy-human coding, we can produce summary tables that make the results easier to read.

These tables answer slightly different questions:

- **Concatenated summary:** if we pool all coded examples across all documents together, how well did the LLM perform for each construct?
- **Per-document summary:** how well did the LLM perform within each individual document?
- **Weighted summary:** what does performance look like across documents when documents with more coded material count more heavily?

### Important note about the metrics

This is an open-ended coding task, not a standard classification task with true negatives.

We observe:
- True positives (TP)
- False positives (FP)
- False negatives (FN)

But we do **NOT** have true negatives (TN), so:
- Specificity is not meaningful
- ROC AUC is not appropriate

**A note on Cohen's Kappa.** Cohen's Kappa adjusts agreement for chance, but it requires a true-negative count. Because open-ended span coding has no observable true negatives (TN = 0 by convention), kappa is biased downward and is not directly comparable to kappa values from fixed-item rating tasks. For this reason, kappa is **omitted from the summary tables by default**. We report **PABAK** (prevalence-adjusted bias-adjusted kappa) instead — it is more stable when prevalence is low and is a better fit for this kind of analysis. If you want kappa anyway for diagnostic purposes, pass `include_cohens_kappa=True` to `compute_summary_tables(...)`.

Reported metrics: precision, recall (sensitivity), F1, PABAK, and PR-AUC.

### 4.1 Row-Level Comparison DataFrame

The raw DataFrame below has one row per coded instance across all documents. Each row represents a single quote that was identified by at least one coder, with the full quote text, character indices, and TP/FP/FN indicators. This is the most granular view and is useful for inspecting specific mismatches — for example, filtering to `status == 'llm_only'` to review what the LLM found that the dummy human missed.

In [10]:
comparison_table = format_comparison_table(comparison_results)
comparison_table

Comparison Table
----------------
One row per coded instance across all interviews. Each row represents a quote
identified by at least one coder, with its classification and match details.

Columns:
  doc_id           : interview identifier
  construct        : psychological construct the instance belongs to
  status           : matched (TP), human_only (FN), or llm_only (FP)
  human_quote      : quote extracted by the human coder
  llm_quote        : quote extracted by the LLM
  human_indices    : character-level start:end indices of the human quote in the source text
  llm_indices      : character-level start:end indices of the LLM quote in the source text
  paraphrase       : True if the matched quotes differ meaningfully in wording
  span_overlap     : Jaccard overlap between human and LLM character spans (matched rows only)
  match_confidence : LLM matcher confidence that the two quotes refer to the same passage
  tp/fp/fn         : binary indicators for this row's contribution to

,doc_id,construct,status,human_quote,llm_quote,human_indices,llm_indices,paraphrase,span_overlap,match_confidence,tp,fp,fn
0,anxiety_4ssCr4ck,anxiety,matched,Is this anxiety?,Is this anxiety?,0:16,0:16,False,1.0,1.0,1,0,0
1,anxiety_4ssCr4ck,anxiety,matched,people that really have anxiety,people that really have anxiety,71:102,71:102,False,1.0,1.0,1,0,0
2,anxiety_4ssCr4ck,anxiety,matched,anxious,anxious,299:306,299:306,False,1.0,1.0,1,0,0
3,anxiety_4ssCr4ck,depressed_mood,matched,crying,crying aswell,372:378,372:385,False,0.461538,1.0,1,0,0
4,anxiety_4ssCr4ck,panic,matched,made me literally panic,made me literally panic,254:277,254:277,False,1.0,1.0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,depression_scrappydoosghost,shame,matched,I SHOULD die,"and the fact is, I SHOULD die.",425:437,408:438,True,0.4,1.0,1,0,0
296,depression_scrappydoosghost,shame,matched,I hate me so much,I hate me so much,453:470,453:470,False,1.0,1.0,1,0,0
297,depression_scrappydoosghost,shame,matched,what do I deserve? Death?,so what do I deserve? Death?,597:622,594:622,True,0.892857,1.0,1,0,0
298,depression_scrappydoosghost,shame,llm_only,NaN,I don’t deserve any of this I swear- what did ...,NaN,472:523,None,NaN,NaN,0,1,0


In [11]:
per_interview, concatenated, weighted_summary = compute_summary_tables(comparison_table)

### 4.2 Concatenated Metrics (primary view)

One row per construct, with counts pooled across **all documents** before metrics are computed. Think of this as treating the entire dataset as a single document. This table answers: *overall, across all documents, how did the LLM perform on each construct?*

The `interviews_with_construct [p5-p95]` column shows how many documents contained at least one instance of the construct, along with the 5th and 95th percentile of instance counts — giving a sense of how common and variable the construct is across your data.

In [12]:
format_concatenated(concatenated)

Concatenated Metrics
--------------------
One row per construct. TP/FP/FN counts are pooled across all interviews before
metrics are computed — treats the entire dataset as a single document.

Columns:
  tp/fp/fn                          : total counts pooled across all interviews
  sensitivity                       : TP / (TP + FN) — proportion of human-coded instances the LLM found
  precision                         : TP / (TP + FP) — proportion of LLM-coded instances that were correct
  f1                                : harmonic mean of sensitivity and precision
  pabak                             : prevalence-adjusted kappa; more stable when construct prevalence is low
  pr_auc                            : area under precision-recall curve; ranks LLM predictions by match confidence
  interviews_with_construct [p5-p95]: number of interviews containing the construct, with 5th-95th percentile of instance counts



,construct,tp,fp,fn,sensitivity,precision,f1,pabak,pr_auc,interviews_with_construct [p5-p95]
0,agitation,6,3,0,1.00,0.67,0.80,0.33,1.00,2 [0.00-0.00]
1,anhedonia,21,9,8,0.72,0.70,0.71,0.11,1.00,12 [0.00-4.40]
2,anxiety,61,5,10,0.86,0.92,0.89,0.61,1.00,34 [0.00-4.00]
3,depressed_mood,34,8,8,0.81,0.81,0.81,0.36,1.00,22 [0.00-3.40]
4,guilt,11,2,0,1.00,0.85,0.92,0.69,1.00,9 [0.00-1.00]
5,panic,16,7,1,0.94,0.70,0.80,0.33,1.00,15 [0.00-2.00]
6,rumination,22,4,2,0.92,0.85,0.88,0.57,1.00,16 [0.00-2.40]
7,shame,47,6,7,0.87,0.89,0.88,0.57,1.00,25 [0.00-4.00]
8,trauma_and_ptsd,2,0,0,1.00,1.00,1.00,1.00,—,2 [0.00-0.00]
9,Overall,220,44,36,0.86,0.83,0.85,0.47,1.00,53 [0.00-3.00]


### Alternate Summary Tables

The two tables below offer complementary views to the concatenated summary above. Use them when you need to dig into individual-document performance or want results that down-weight documents with very little coded material.

#### Per-Document Metrics

One row per (document, construct) combination. Shows raw TP/FP/FN counts and all metrics broken down by document. Use this to identify whether performance issues are consistent across documents or driven by specific ones. Constructs that do not appear in a given document are absent from this table, they do not appear as zero rows. 

In [13]:
format_per_interview(per_interview)

Per-Interview Metrics
---------------------
One row per (interview, construct) combination. Constructs that did not appear
in a given interview are absent — they do not appear as zero rows.

Columns:
  doc_id       : interview identifier
  construct    : psychological construct
  tp/fp/fn     : true positives, false positives, false negatives for this interview-construct
  union        : total coded instances (TP + FP + FN)
  sensitivity  : TP / (TP + FN)
  precision    : TP / (TP + FP)
  f1           : harmonic mean of sensitivity and precision
  pabak        : prevalence-adjusted kappa
  pr_auc       : area under precision-recall curve; NaN where insufficient label classes



,doc_id,construct,tp,fp,fn,union,sensitivity,precision,f1,pabak,pr_auc
0,anxiety_4ssCr4ck,anxiety,3,0,0,3,1.0,1.0,1.00,1.0,NaN
1,anxiety_4ssCr4ck,depressed_mood,1,0,0,1,1.0,1.0,1.00,1.0,NaN
2,anxiety_4ssCr4ck,panic,2,0,0,2,1.0,1.0,1.00,1.0,NaN
3,anxiety_4ssCr4ck,shame,1,0,0,1,1.0,1.0,1.00,1.0,NaN
4,anxiety_An0nus3r123,anhedonia,0,1,0,1,NaN,0.0,0.00,-1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
132,depression_nefariousbeliever,depressed_mood,1,0,0,1,1.0,1.0,1.00,1.0,NaN
133,depression_nefariousbeliever,shame,2,0,0,2,1.0,1.0,1.00,1.0,NaN
134,depression_scrappydoosghost,anhedonia,1,0,0,1,1.0,1.0,1.00,1.0,NaN
135,depression_scrappydoosghost,shame,4,1,0,5,1.0,0.8,0.89,0.6,1.0


#### Weighted Summary

One row per construct. Metrics are first computed per document, then summarized as **weighted median [min–max]** across documents, where each document is weighted by its union size (TP + FP + FN) — so documents with more coded instances contribute more to the median. This table answers: *what does performance typically look like at the individual document level?* The `[min–max]` brackets show the range across documents, useful for spotting constructs with highly variable performance.



In [14]:
format_weighted_summary(weighted_summary)

Weighted Summary
----------------
One row per construct. Metrics are computed per interview first, then summarized
as weighted median [min–max] across interviews, weighted by union size (TP+FP+FN).

Columns:
  tp/fp/fn                          : total true positives, false positives, false negatives across all interviews
  sensitivity                       : TP / (TP + FN) — proportion of human-coded instances the LLM found
  precision                         : TP / (TP + FP) — proportion of LLM-coded instances that were correct
  f1                                : harmonic mean of sensitivity and precision
  pabak                             : prevalence-adjusted kappa; more stable when construct prevalence is low
  pr_auc                            : area under precision-recall curve; ranks LLM predictions by match confidence
  interviews_with_construct [p5-p95]: number of interviews containing the construct, with 5th-95th percentile of instance counts



,construct,tp,fp,fn,sensitivity,precision,f1,pabak,pr_auc,interviews_with_construct [p5–p95]
0,anxiety,61,5,10,1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [-1.00–1.00],1.00 [1.00–1.00],34 [0.00–4.00]
1,depressed_mood,34,8,8,0.83 [0.33–1.00],1.00 [0.00–1.00],0.86 [0.00–1.00],0.50 [-1.00–1.00],1.00 [1.00–1.00],22 [0.00–3.40]
2,panic,16,7,1,1.00 [0.50–1.00],1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [-1.00–1.00],1.00 [1.00–1.00],15 [0.00–2.00]
3,shame,47,6,7,1.00 [0.00–1.00],1.00 [0.00–1.00],0.89 [0.00–1.00],0.60 [-1.00–1.00],1.00 [1.00–1.00],25 [0.00–4.00]
4,anhedonia,21,9,8,0.75 [0.00–1.00],0.60 [0.00–1.00],0.60 [0.00–1.00],-0.14 [-1.00–1.00],1.00 [1.00–1.00],12 [0.00–4.40]
5,rumination,22,4,2,1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [-1.00–1.00],1.00 [1.00–1.00],16 [0.00–2.40]
6,trauma_and_ptsd,2,0,0,1.00 [1.00–1.00],1.00 [1.00–1.00],1.00 [1.00–1.00],1.00 [1.00–1.00],—,2 [0.00–0.00]
7,guilt,11,2,0,1.00 [1.00–1.00],1.00 [0.00–1.00],1.00 [0.00–1.00],1.00 [-1.00–1.00],—,9 [0.00–1.00]
8,agitation,6,3,0,1.00 [1.00–1.00],0.62 [0.62–1.00],0.77 [0.77–1.00],0.25 [0.25–1.00],1.00 [1.00–1.00],2 [0.00–0.00]
9,Overall,220,44,36,1.00 [0.00–1.00],1.00 [0.00–1.00],0.91 [0.00–1.00],0.67 [-1.00–1.00],1.00 [1.00–1.00],53 [0.00–3.00]


---

# Part 2 — Real Human Data Workflow

In a typical research workflow you will compare the LLM's coding against **real human-coded data**, not against a second LLM run. This part walks through that workflow end-to-end.

The steps are the same as Part 1, with one key difference: instead of generating a stand-in human coding via a second LLM run, you load human-coded data from a CSV or xlsx file (e.g., a Dedoose excerpt export, or any other tabular human-coding format) using `load_human_coding(...)`.

## Step 1: LLM Coding

This is the same LLM-coding step as in Part 1.

> **If you already ran the LLM in Part 1 and want to reuse those results**, skip this cell and continue to Step 2 — Part 2 below uses `results_llm` from above.
>
> **Run this cell only if you want a fresh, independent LLM run for Part 2** (e.g., a different model, a different output directory, or you skipped Part 1 entirely).

In [ ]:
input_data = "" # path to the csv or directory the llm will be coding.

In [ ]:
# Run this only if you want a fresh LLM run for Part 2; otherwise skip and reuse `results_llm` from Part 1.

real_analyzer = LLMTrackerAnalyzer(
    api_key=api_key,
    model_name=llm_model,
)

results_llm, metadata_llm, errors_llm = real_analyzer.analyze_csv(
    csv_path=input_data,
    text_column="post",
    subreddit_column="subreddit",
    author_column="author",
    codebook_path=codebook_path,
    output_dir="LLM_coding_real",  # timestamp suffix added automatically
)
format_coding_table(results_llm)

### Alternative: directory of `.txt` files

Same as in Part 1 — if your input is a folder of `.txt` files, swap `analyze_csv` for `analyze_directory`. Cell below left commented as a reference.

In [ ]:
# Alternative: analyze a directory of .txt files instead of a CSV.

# input_dir = ""  # path to your directory of .txt files

# results_llm, metadata_llm, errors_llm = real_analyzer.analyze_directory(
#     input_dir=input_dir,
#     codebook_path=codebook_path,
#     output_dir="LLM_coding_real",  # timestamp suffix added automatically
# )

## Step 2 : Loading Human-Coded Data

Real human coding typically lives in a tabular file (CSV, TSV, or xlsx) where each row is a single coded excerpt. `load_human_coding(...)` reads that file and converts it into the same internal `AnalysisResult` format that `analyze_csv` / `analyze_directory` produce, so it can be passed straight to the comparator.

The loader is **generic** — it works with any tabular human-coding source as long as you tell it which columns to use:

| Argument            | Default                       | What it identifies                                                    |
|---------------------|-------------------------------|------------------------------------------------------------------------|
| `doc_id_col`        | `"Media Title"`               | Column whose value matches the LLM `document_id` (filename / CSV row) |
| `quote_col`         | `"Excerpt Copy"`              | The verbatim text excerpt the human selected                           |
| `range_col`         | `"Excerpt Range"`             | Character-level start–end indices into the source document             |
| `construct_col`     | `"Codes Applied Combined"`    | Code(s) applied to the excerpt; multiple codes are split on the separator |
| `range_format`      | `"dash"` (e.g. `"858-1159"`)  | `"dash"` for Dedoose-style; `"colon"` for `start:end` format           |
| `construct_separator` | `","`                       | Used to split multi-code cells into individual constructs              |

The defaults match a Dedoose **Excerpt Export** out of the box, so a Dedoose user only needs `load_human_coding(path)`. For other tabular sources, override the column-name kwargs accordingly.

> **Important — document-id alignment.** The values in `doc_id_col` must match the document IDs used in your LLM run (typically the source filename without extension, or the CSV-row identifier built from `subreddit_column` + `author_column`). If they don't align, the comparator won't pair instances correctly.

In [ ]:
human_csv_path = ""  # path to your human-coded CSV/xlsx export

# Generic example — override only the kwargs that differ from the Dedoose defaults.
# For a non-Dedoose source, pass the column names explicitly, e.g.:
#
# human_results = load_human_coding(
#     human_csv_path,
#     doc_id_col="participant_id",
#     quote_col="excerpt_text",
#     range_col="char_range",
#     construct_col="code",
#     range_format="colon",          # if your ranges look like "858:1159"
#     construct_separator=";",        # if codes are separated by something else
# )

human_results = load_human_coding(human_csv_path)

### Optional: Validate human-coded constructs against the codebook

Loading does **not** validate that the constructs found in the human file all exist in your codebook — that's deliberate, so you can inspect the file first. Run the cell below if you want a sanity check before comparison; it returns a `ValidationReport` that lists any unknown constructs by document. With `strict=True` it raises an error if any unknown constructs are found.

In [ ]:
validation_report = validate_against_codebook(
    results=human_results,
    codebook=codebook_path,
    strict=False,
)
print(validation_report)

## Step 3: Comparison

Same comparison step as Part 1, but now we pass the real human results in place of the dummy ones.

In [ ]:
real_comparator = LLMTrackerComparer(
    api_key=api_key,
    match_model=llm_model,
)

comparison_results_real = real_comparator.compare_results(
    human_results=human_results,
    llm_results=results_llm,
    output_dir="real_comparison_run",  # optional; saves JSON comparison files
)

## Step 4: Summary Tables

Same four summary tables as Part 1, in the same order: row-level table → concatenated → alternate (per-document + weighted). See the metrics note in Part 1 Step 4 for details on PABAK vs Cohen's Kappa.

### 4.1 Row-Level Comparison DataFrame

One row per coded instance across all documents — useful for inspecting specific mismatches.

In [ ]:
comparison_table_real = format_comparison_table(comparison_results_real)
comparison_table_real

In [ ]:
per_interview_real, concatenated_real, weighted_summary_real = compute_summary_tables(comparison_table_real)

### 4.2 Concatenated Metrics (primary view)

Counts pooled across all documents — overall LLM performance per construct against your real human coders.

In [ ]:
format_concatenated(concatenated_real)

### Alternate Summary Tables

#### Per-Document Metrics

Performance broken down by document — useful for spotting documents that drive overall numbers.

In [ ]:
format_per_interview(per_interview_real)

#### Weighted Summary

Per-document metrics summarized as weighted median [min–max] across documents.

In [ ]:
format_weighted_summary(weighted_summary_real)